**BLOCK 0 - Install Required Python Packages**<br><br>
This block of code installs the libraries that you will use throughout the lab. It may take a few moments for the libraries to install. Be patient!

In [ ]:
# ----------------------
# Install pandas (data tables)
# ----------------------
!python -m pip install pandas --quiet

# ----------------------
# Install matplotlib (plotting)
# ----------------------
import sys
!{sys.executable} -m pip install matplotlib --quiet

# ----------------------
# Install SciPy (statistics + scientific tools)
# ----------------------
import sys
!{sys.executable} -m pip install scipy --quiet

# ----------------------
# Install scikit-learn (machine learning)
# ----------------------
import sys
!{sys.executable} -m pip install scikit-learn --quiet

print("🎉 All required packages are now installed! You're ready for Block 1.")

**BLOCK 1 - Load Data and Extract Features** <br><br>
In this block, you will: <br>
Load the dataset from the original research report and define: <br>
- Target value (the value you want to predict: Photostability)
- Feature names (the column header in the csv file gives the name for each chemical feature)
- Numerical values for each feature <br><br> **Why is this step necessary?**<br>Your computer does not automatically understand the contents of your spreadsheet. It is your responsibility to tell the computer what's what!

Before running the cell, open the file OligomerFeatures.csv in Jupyter by double clicking on the file. The first column names each molecule in the dataset. Column G is your target value. Columns H through EU are chemical features. <br>

- Column G (Index 5): Photostability

In [ ]:
# ===========================================================
# BLOCK 1 — Load Data and Extract Features
# ===========================================================

import pandas as pd
import numpy as np

# ✏️ STUDENT TASK:
# Upload the CSV file into Jupyter Notebook using the upload files button (Up arrow, top left corner).
# Make sure the CSV file is in the same folder as your notebook.
# If your file has a different name, change it below:
df = pd.read_csv("OligomerFeatures.csv")

# Number of molecules in the dataset. 
molecs = 40  

# -----------------------------------------------------------
# Extract feature names and feature values
# -----------------------------------------------------------
# Columns H through EU correspond to index 7 through 143.
# We use .values to get the column NAMES
# and .iloc to get the numerical DATA.
# -----------------------------------------------------------

first_feature_col = 7     # H
last_feature_col  = 143    # EU

data_labels = np.array(df.columns.values[first_feature_col:last_feature_col])
all_features = np.array(df.iloc[:molecs, first_feature_col:last_feature_col])

# -----------------------------------------------------------
# Extract target values. These are the target values we want to predict. 
# -----------------------------------------------------------
# ✏️ STUDENT TASK:
# For each notebook, apply the target (Photostability) and target index (5)
Type_Target_Here = np.array(df.iloc[:molecs, Type_Target_Index_Here])

print("🎉 All required data and features have been loaded! You're ready for Block 2.")

# ❓ Reflection:
# 1. How many features are in your dataset?
# 2. Why is it important to separate the target from the features?

**Block 2  - Define regression models** <br>
<br>
In these blocks, you will define a mathematical function that: <br>
- Fits a linear regression model for predicting a target property (Photostability) based on selected features
- Uses the linear regression model to predict the target value (Photostability)
- Analyzes the goodness of fit for your regression model using R^2 and adjusted R^2 <br>
  <br>
The regression models will not yet be reported. You are just setting up the mathematical function for the subsequent steps. <br><br>
  
**R^2** tells you how well the model fits the data. A value of 1 would indicate all data is explained by the model. A value of 0 would indicate the model is not better than simply predicting the average value of a target. <br> <br>
**Adjusted R^2** is a modified version of R^2 which penalizes a model for unnecessarily incoporating too many features in its prediction. The use of too many features can lead to overfitting. Although R^2 often improves when more features are used, a model trained on more features does not necessarily provide improved predictive accuracy. 

In [ ]:
# ===========================================================
# BLOCK 2 — Define regression models
# ===========================================================

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


def evaluate_model(X, y, feature_names):
    """
    INPUTS:
    X  → Feature matrix (rows = molecules, columns = descriptors)
    y  → Target property you want to predict
    """

    # ----------------------------------------------
    # 1. Fit the linear regression model
    # ----------------------------------------------
    model = LinearRegression().fit(X, y)

    # ----------------------------------------------
    # 2. Predict y values using the trained model
    # ----------------------------------------------
    y_pred = model.predict(X)

    # ----------------------------------------------
    # 3. Compute R² (goodness of fit)
    # ----------------------------------------------
    r2 = r2_score(y, y_pred)

    # ----------------------------------------------
    # 4. Compute adjusted R²
    #    Formula penalizes excessive numbers of features.
    # ----------------------------------------------
    adj_r2 = 1 - (1 - r2) * (len(y) - 1) / (len(y) - X.shape[1] - 1)

    # ----------------------------------------------
    # 5. Return results in a tidy structure
    # ----------------------------------------------
    return {
        "features": feature_names,
        "model": model,
        "r2": r2,
        "adj_r2": adj_r2
    }

# ❓ Reflection:
# 3. What is the value of adjusted R² when comparing models with different numbers of features?

print("🎉 Linear regression models have been trained! You're ready for Block 3.")

**Block 3 - Select Top Features via Univariate Regression** <br>
<br>
In this block, you will use the mathematical function defined in the previous block to: <br>
- Iteratively perform **univariate regression** to test how each feature individually affects the target function (Photostability)
- Quantify and rank the goodness of fit via R^2 and Adjusted R^2
- Select the top features for further analysis <br> <br>

**Selecting top features** can be a useful data reduction strategy for workflows where very large numbers of features are available. After univariate regression, one can choose to move forward with features which individually influence the target function. This downselection can be useful during multivariate regression to:
1) Reduce computational time
2) Avoid overfitting
3) Focus on chemically meaningful features


In [ ]:
# ===========================================================
# BLOCK 3 — Select Top Features via Univariate Regression
# ===========================================================


import pandas as pd

#Select the number of top features to use for subsequent analysis using k=value
def univariate_top_k(X, y, labels, k=46):
    """
    INPUTS:
    X      → Feature matrix (rows = molecules, columns = descriptors)
    y      → Target property
    labels → List of feature names corresponding to columns in X
    k      → Number of top features to return

    RETURNS:
    top_k  → List of dictionaries containing top k feature models
    report → Pandas DataFrame tabulating R² and adjusted R² for top k features
    """

    results = []
    for i, lab in enumerate(labels):
        # Extract single feature as 2D array
        Xi = X[:, i].reshape(-1, 1)
        # Evaluate a linear regression for this feature
        res = evaluate_model(Xi, y, [lab])
        results.append(res)

    # Sort features by absolute R² (strongest correlations first)
    results = sorted(results, key=lambda x: abs(x["r2"]), reverse=True)
    top_k = results[:k]

    # Create a nice table to inspect
    report = pd.DataFrame({
        "Feature": [m["features"][0] for m in top_k],
        "R2": [round(m["r2"], 3) for m in top_k],
        "Adjusted R2": [round(m["adj_r2"], 3) for m in top_k]
    })

    return top_k, report
print("🎉 Univariate regression complete. Top features are tabulated. Proceed to Block 4 to run multivariable regressions!")

# -----------------------------------------------------------
# Run univariate regression and select top features
# -----------------------------------------------------------
#
# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here

top_uni, uni_report = univariate_top_k(all_features, Record_Target_Here, data_labels, k=46)
selected_features = [m["features"][0] for m in top_uni]

# Display the results as a visually appealing table in Jupyter
uni_report.style.background_gradient(cmap='Blues')

# ❓ Reflection:
# 4. When would you expect more features to help a regression analysis? When would you expect 
#    more features to hurt a regression analysis?

**Block 4 - Multivariate Regression** <br>
<br>
In this block, you will: <br>
- Evaluate combinations of 2-4 features
- Fit a linear regression model to each combination
- Rank generated models based on adjusted R^2

**Why is multivariate regression useful?**<br>
  Sometimes one variable is not sufficient to completely define a system of interest. For example, for reaction development, both steric and electronic parameters are often important to dictate reactivity. 

  **Caution:** This block of code will take several minutes (up to 5) to run. **DO NOT** refresh the page or the calculations will start over. Wait until the Multivariate Regression Complete notification is displayed before proceeding. 
<br> <br>
This is an example of why it is useful to downselect features, which we did in the previous step. Multivariate regression experiments quickly become computationally expensive. 
<br> <br>
For reference, scanning tetravariate regressions across 10 features is **210 experiments**. Screening 50 features is **230,300 experiments**. Screening 100 features is **3,921,225 experiments**! Number of features is always a cost-benefit analysis of time versus meaningful improvement in model generation.

In [ ]:
# ===========================================================
# BLOCK 4 — Multivariate Regression
# ===========================================================

from itertools import combinations
from IPython.display import display

def multivariate_top_k(X, y, labels, selected_labels, n_vars=2, k=5):
    """
    INPUTS:
    X               → Full feature matrix
    y               → Target property
    labels          → Names of all features in X
    selected_labels → Subset of features to consider (from univariate step)
    n_vars          → Number of features per combination (2,3,4)
    k               → Number of top models to report

    RETURNS:
    top_k → List of dictionaries with top models
    report → Pandas DataFrame showing features, R², and adjusted R²
    """
    # Convert feature names to their column indices
    indices = [np.where(labels == lab)[0][0] for lab in selected_labels]
    results = []

    # Generate all combinations of n_vars features
    for combo in combinations(indices, n_vars):
        Xsub = X[:, combo]
        labs = labels[list(combo)]
        res = evaluate_model(Xsub, y, labs)
        results.append(res)

    # Sort by adjusted R² (highest first)
    results = sorted(results, key=lambda x: x["adj_r2"], reverse=True)
    top_k = results[:k]

    # Create a table for display
    report = pd.DataFrame({
        "Features": [", ".join(m["features"]) for m in top_k],
        "R2": [round(m["r2"], 3) for m in top_k],
        "Adjusted R2": [round(m["adj_r2"], 3) for m in top_k]
    })

    return top_k, report

# -----------------------------------------------------------
# Run multivariate models
# -----------------------------------------------------------

# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here

# Bivariate (2-feature) models
top_bi, bi_report = multivariate_top_k(all_features, Record_Target_Here, data_labels,
                                       selected_features, n_vars=2, k=5)

# Trivariate (3-feature) models
top_tri, tri_report = multivariate_top_k(all_features, Record_Target_Here, data_labels,
                                         selected_features, n_vars=3, k=5)

# Tetravariate (4-feature) models
top_tetra, tetra_report = multivariate_top_k(all_features, Record_Target_Here, data_labels,
                                             selected_features, n_vars=4, k=5)

print("🎉 Multivariate regression complete. Top models are tabulated. Proceed to Block 5 to visualize best models!")


# -----------------------------------------------------------
# Display tables with visual gradients and print headings
# -----------------------------------------------------------

print("🎉 Top 5 Bivariate Models")
display(bi_report.style.background_gradient(cmap='Blues'))

print("🎉 Top 5 Trivariate Models")
display(tri_report.style.background_gradient(cmap='Greens'))

print("🎉 Top 5 Tetravariate Models")
display(tetra_report.style.background_gradient(cmap='Purples'))

# ❓ Reflection:
# 5. How does adding more features affect R²?
# 6. How does adjusted R² typically compare to R²?
# 7. Do you notice any descriptors that appear repeatedly in the top models?



**Block 5 - Visualize Top Models** <br>
<br>
In this block, you will: <br>
- Plot the top univariate linear regression to visualize the variance
- Plot the multivariate analyses in the form of a **parity plot** to visualize how well the model predicts the actual data

**Why is a parity plot useful for data visualization?**<br>
Multivariate models become hard to visualize as number of features increase, since the plots are not merely 2-dimensional. Parity plots simplify multidimensional data into two dimensions to allow easy visualization of model performance. Notice that the axes are predicted versus actual target value, not specific features. 

In [ ]:
# ===========================================================
# BLOCK 5 — Visualize Top Models
# ===========================================================


import matplotlib.pyplot as plt

def plot_model(model_info, X, y, labels):
    """
    Plot either:
        • Univariate regression (scatter plot + best-fit line)
        • Multivariate regression (predicted vs actual)
    depending on the number of features in the model.
    """

    model = model_info["model"]
    feature_names = model_info["features"]

    # -------------------------------------------------------
    # Case 1: Univariate model → scatter + regression line
    # -------------------------------------------------------
    if len(feature_names) == 1:
        idx = np.where(labels == feature_names[0])[0][0]
        Xi = X[:, idx]

        plt.figure(figsize=(6, 4))
        plt.scatter(Xi, y, alpha=0.7)

        # Regression line
        xline = np.linspace(min(Xi), max(Xi), 200).reshape(-1, 1)
        yline = model.predict(xline)
        plt.plot(xline, yline, color='red')

        plt.xlabel(feature_names[0])
# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here
        plt.ylabel("Record_Target_Here")
        plt.title(f"Univariate Model: {feature_names[0]}")
        plt.tight_layout()
        plt.show()
        return

    # -------------------------------------------------------
    # Case 2: Multivariate model → predicted vs actual plot
    # -------------------------------------------------------
    col_indices = [np.where(labels == n)[0][0] for n in feature_names]
    Xsub = X[:, col_indices]
    y_pred = model.predict(Xsub)

    plt.figure(figsize=(6, 4))
    plt.scatter(y, y_pred, alpha=0.7)
    plt.plot([min(y), max(y)], [min(y), max(y)], color='gray', linestyle='--')

# ✏️ STUDENT TASK: 
#Label your x-axis Actual Photostability. Label your y-axis Predicted Photostability.
    plt.xlabel("X-axis_Title")
    plt.ylabel("Y-axis_Title")
    plt.title(f"{len(feature_names)}-Feature Model: {', '.join(feature_names)}")
    plt.tight_layout()
    plt.show()
print("🎉 Visualization complete. Proceed to Block 6!")

# -----------------------------------------------------------
# Visualize the top models from each category
# -----------------------------------------------------------

# ✏️ STUDENT TASK:
# For each model, change the target (SO, T80, or Photostability)
print("📈 Plotting Best Univariate Model...")
plot_model(top_uni[0], all_features, T80, data_labels)

print("📈 Plotting Best Bivariate Model...")
plot_model(top_bi[0], all_features, T80, data_labels)

print("📈 Plotting Best Trivariate Model...")
plot_model(top_tri[0], all_features, T80, data_labels)

print("📈 Plotting Best Tetravariate Model...")
plot_model(top_tetra[0], all_features, T80, data_labels)

# ❓ Reflection:
# 8. Based on visualization of your univariate regression, do you think a 
#    univariate linear regression is sufficient to predict the target value
#    or new OPV candidates? Why or why not? 
# 
# 9. The dashed line on the parity plots has a slope of 1. It is not a regression line. 
#    What does the dashed line represent? 
#
# 10. If a point on the parity plot is above the dashed line, what does this tell you
#     about the model's prediction for that point? Conversely, if a point is below the 
#     dashed line, what does this mean? 


**Block 6 - Extract Linear Equations & Interpret Coefficients** <br>
<br>
In this block, you will: <br>
- Scale features to allow for head-to-head comparison of coefficients
- Report the linear regression equation for each top model using unscaled and scaled features
- Use the scaled coefficients to rationalize the relative contributions of each feature in a    model

Some features have a large range of possible values, while other features may only be present in a narrow range. This situation may inflate/deflate coefficients, making a feature seem more/less important in the model. **Scaling** makes coefficients in the model interpretable; a high coefficient in a scaled model means the feature is more significantly influencing the model than a feature with a smaller coefficient. 

In [ ]:
# ===========================================================
# BLOCK 6 — Extract Linear Equations & Interpret Coefficients
# ===========================================================

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

def compare_scaled_unscaled(model_info, X, y, labels):
    """
    Returns:
        eq_unscaled  → equation using original features
        eq_scaled    → equation using standardized features
        table        → DataFrame with Feature, Coefficient, Scaled Coefficient
    """

    features = model_info["features"]

    # Get indices for selected features
    col_indices = [np.where(labels == f)[0][0] for f in features]
    Xsub = X[:, col_indices]

    # ----------------------------
    # Unscaled model
    # ----------------------------
    model_unscaled = LinearRegression().fit(Xsub, y)
    coefs_unscaled = model_unscaled.coef_
    intercept_unscaled = model_unscaled.intercept_
# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here
    terms_unscaled = [f"{round(c, 3)}*{f}" for c, f in zip(coefs_unscaled, features)]
    eq_unscaled = f"Record_Target_Here = {round(intercept_unscaled, 3)} + " + " + ".join(terms_unscaled)

    # ----------------------------
    # Scaled model
    # ----------------------------
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(Xsub)

    model_scaled = LinearRegression().fit(X_scaled, y)
    coefs_scaled = model_scaled.coef_
    intercept_scaled = model_scaled.intercept_
# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here
    terms_scaled = [f"{round(c, 3)}*{f}" for c, f in zip(coefs_scaled, features)]
    eq_scaled = f"Record_Target_Here = {round(intercept_scaled, 3)} + " + " + ".join(terms_scaled)

    # ----------------------------
    # Combined table
    # ----------------------------
    table = pd.DataFrame({
        "Feature": features,
        "Coefficient": [round(c, 3) for c in coefs_unscaled],
        "Scaled Coefficient": [round(c, 3) for c in coefs_scaled]
    })

    table = table.style.background_gradient(cmap="coolwarm", subset=["Scaled Coefficient"])

    return eq_unscaled, eq_scaled, table

    # ===========================================================
#   Compare scaled vs unscaled models
# ===========================================================
# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here
def display_model(name, model_info):
    eq_unscaled, eq_scaled, table = compare_scaled_unscaled(
        model_info, all_features, Record_Target_Here, data_labels
    )

    print(f"\n📘 {name} — Unscaled Equation:")
    print(eq_unscaled)

    print(f"\n📗 {name} — Scaled Equation:")
    print(eq_scaled)

    display(table)


# ---- Run for each model ----
display_model("Univariate Model", top_uni[0])
display_model("Bivariate Model", top_bi[0])
display_model("Trivariate Model", top_tri[0])
display_model("Tetravariate Model", top_tetra[0])

# ❓ Reflection:
# 11. Look at the sign of each coefficient. What does it mean if a coefficient is positive 
#     versus negative? 
# 
# 12. Consider the values for the unscaled coefficient versus the scaled coefficients. 
#     Provide an example where the two values differ significantly, then explain why scaling 
#     was required to meaningfuly interpret the relative influence of each feature in the model. 
#
# 13. Which features appear to be most important for predicting Photostability?

**Block 7 - Train and Test** <br>
<br>
In this block, you will: <br>
- Split molecules with known target values into **training** and **test** sets
- Use the top models trained on the **training set** to predict the target function for the **test set**
- Analyze Train and Test R^2 values to reveal why increasing number of features in a model may lead to **overfitting**

**Overfitting** is a problem in machine learning where increasing the number of features used to train a model can enhance R^2 value for the training set. However, improvement in training set R^2 does not always mean improvement in the ability of the model to predict function for a new test set. 

In [ ]:
# ===========================================================
# BLOCK 7 — Train/Test Split + Visualization
# ===========================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import numpy as np

def evaluate_train_test(model_info, X, y, labels, test_size=0.25):
    """
    Train/test evaluation for a given model.
    """

    features = model_info["features"]
    indices = [np.where(labels == f)[0][0] for f in features]
    Xsub = X[:, indices]

    X_train, X_test, y_train, y_test = train_test_split(
        Xsub, y, test_size=test_size, random_state=42
    )

    model = LinearRegression().fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test


# -----------------------------------------------------------
# Evaluate models
# -----------------------------------------------------------

model_names = ["Univariate", "Bivariate", "Trivariate", "Tetravariate"]
model_list = [top_uni[0], top_bi[0], top_tri[0], top_tetra[0]]

train_scores = []
test_scores = []


# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here
print("📊 Train/Test Results:")
for name, model in zip(model_names, model_list):
    r2_train, r2_test = evaluate_train_test(model, all_features, Record_Target_Here, data_labels)
    
    train_scores.append(r2_train)
    test_scores.append(r2_test)
    
    print(f"\n{name} Model")
    print(f"Training R²: {round(r2_train, 3)}")
    print(f"Test R²:     {round(r2_test, 3)}")


# -----------------------------------------------------------
# Visualization
# -----------------------------------------------------------

plt.figure(figsize=(7,5))

plt.plot(model_names, train_scores, marker='o', label="Training R²")
plt.plot(model_names, test_scores, marker='o', label="Test R²")

plt.xlabel("Model Complexity (Number of Features)")
plt.ylabel("R² Score")
plt.title("Training vs Test Performance")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# ❓ Reflection:
# 14. As you increase from univariate to tetravariate models, it is expected that Training R^2
#     will increase. What trend do you observe for the Test R^2? 
#     What does this trend in Test R^2 tell you about the ideal number of features for this model? 

**Block 8 - Overfitting Example** <br>
<br>
In this block, you will: <br>
- Experiment with models of 1-25 features to see how number of features affects Training R^2 versus Test R^2
- Determine what number of features likely leads to **overfitting** for this dataset
 

In [ ]:
# ===========================================================
# BLOCK 8 — Overfitting Exploration (1 → N Features)
# ===========================================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import numpy as np

def evaluate_k_features(k, X, y, labels, ranked_features):
    """
    Build model using top k features and evaluate performance.
    """

    selected = ranked_features[:k]
    indices = [np.where(labels == f)[0][0] for f in selected]
    Xsub = X[:, indices]

    X_train, X_test, y_train, y_test = train_test_split(
        Xsub, y, test_size=0.25, random_state=42
    )

    model = LinearRegression().fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    return r2_train, r2_test


# -----------------------------------------------------------
# Use ranked features from Block 4
# -----------------------------------------------------------
ranked_features = [m["features"][0] for m in top_uni]

max_k = min(25, len(ranked_features))

train_scores = []
test_scores = []
k_values = list(range(1, max_k + 1))

# -----------------------------------------------------------
# Run models for k = 1 → N
# -----------------------------------------------------------

# ✏️ STUDENT TASK: 
#Apply the target (Photostability) below where it says Record_Target_Here

for k in k_values:
    r2_train, r2_test = evaluate_k_features(
        k, all_features, Record_Target_Here, data_labels, ranked_features
    )
    
    train_scores.append(r2_train)
    test_scores.append(r2_test)


# -----------------------------------------------------------
# Visualization
# -----------------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(k_values, train_scores, label="Training R²")
plt.plot(k_values, test_scores, label="Test R²")

plt.xlabel("Number of Features (k)")
plt.ylabel("R² Score")
plt.title("Overfitting as Model Complexity Increases")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

import pandas as pd

first5_df = pd.DataFrame({
    "Number of Features": list(range(1, 11)),
    "Training R²": [round(r, 3) for r in train_scores[:10]],
    "Test R²": [round(r, 3) for r in test_scores[:10]]
})

display(first5_df)

# ❓ Reflection:
# 15. Explain why increasing the number of features leads to improved Training R^2 value,
#     yet too many features leads to poor Test R^2 values. 
# 16. Rationalize how many features you think is appropriate for this linear regression workflow. 